In [1]:
import pandas as pd
from pathlib import Path

In [2]:
# data path
DATA_PATH = Path(
    r"C:\Users\jchmo\Downloads\dataset_football_matches\big5"
)

In [3]:
# dictionary of the league names
league_names = {
    "E0": "Premier League",
    "D1": "Bundesliga",
    "I1": "Serie A",
    "SP1": "La Liga",
    "F1": "Ligue 1"
}

In [4]:
# load all the data
dataframes = []

for file in sorted(DATA_PATH.glob("*.csv")):

    # eg. E0_2122
    file_name = file.stem

    # split the name to season and league
    league_code, season_code = file_name.split("_")

    # league name
    league = league_names[league_code]

    # 2122 -> 2021/22
    start_year = 2000 + int(season_code[:2])
    end_year = 2000 + int(season_code[2:])

    season = f"{start_year}/{str(end_year)[-2:]}"

    # load CSV
    try:
        df_temp = pd.read_csv(file, encoding="utf-8-sig")
    except UnicodeDecodeError:
        df_temp = pd.read_csv(file, encoding="cp1252")

    # delete empty columns
    df_temp.columns = df_temp.columns.str.strip()

    # add info about league and season
    df_temp["Season"] = season
    df_temp["League"] = league
    df_temp["LeagueCode"] = league_code

    # save the data to dataframe
    dataframes.append(df_temp)

# concatenate all dataframes
df = pd.concat(
    dataframes,
    ignore_index=True,
    sort=False
)

In [9]:
# manage date
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True)
df = df.sort_values("Date").reset_index(drop = True)
print(df["Date"].min())
print(df["Date"].max())
print(df["FTR"].value_counts())


2021-08-06 00:00:00
2026-05-24 00:00:00
FTR
H    3878
A    2769
D    2261
Name: count, dtype: int64


In [10]:
target = "FTR"

metadata_cols = [
    "Div",
    "Date",
    "Time",
    "HomeTeam",
    "AwayTeam",
    "Season",
    "League",
    "LeagueCode",
    "Referee",
]

result_cols = [
    "FTHG",
    "FTAG",
    "FTR",
    "HTHG",
    "HTAG",
    "HTR",
    "HS",
    "AS",
    "HST",
    "AST",
    "HF",
    "AF",
    "HC",
    "AC",
    "HY",
    "AY",
    "HR",
    "AR",
]

In [13]:
def get_points(result, is_home):
    if result == "D":
        return 1
    
    if result == "H":
        return 3 if is_home else 0
    
    if result == "A":
        return 0 if is_home else 3

df["HomePoints"] = df.apply(
    lambda row: get_points(row["FTR"], True),
    axis=1
)

df["AwayPoints"] = df.apply(
    lambda row: get_points(row["FTR"], False),
    axis=1
)

df.groupby("HomeTeam")["HomePoints"].rolling(5)

RollingGroupby [window=5,center=False,axis=0,method=single]